Importation des bibliothèques

In [151]:
import os
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision
import torchvision.transforms as transforms
import torch.optim as optim
from torch.utils.data import random_split

In [152]:
import os
print(os.getcwd())  # donne le répertoire courant

d:\INRIA


Configuration de base

Importation de CIFAR-10

In [153]:
batch_size = 128 #on peut changer la taille mais 128 est bien pour éviter underfitting ou overfitting

transform = transforms.Compose([
    transforms.ToTensor(),                 #Convertir une image en tenseur, met les valeurs des pixels entre 0 et 1
    transforms.Normalize((0.5, 0.5, 0.5),  # Moyenne RGB
                         (0.5, 0.5, 0.5))  # Écart-type RGB; pixels deviennent centrés autour de 0
])


trainset = torchvision.datasets.CIFAR10(root='./data', train=True,
                                         download=True, transform=transform)
trainloader = torch.utils.data.DataLoader(trainset, batch_size=batch_size,
                                          shuffle=True, num_workers=2)

testset = torchvision.datasets.CIFAR10(root='./data', train=False,
                                        download=True, transform=transform)
testloader = torch.utils.data.DataLoader(testset, batch_size=batch_size,
                                         shuffle=False, num_workers=2)

classes = ('plane', 'car', 'bird', 'cat', 'deer',
           'dog', 'frog', 'horse', 'ship', 'truck')


Définition du CNN de base (3 couches)

In [154]:
class SimpleCNN(nn.Module):
    def __init__(self):
        super(SimpleCNN, self).__init__()
        self.conv1 = nn.Conv2d(3, 16, kernel_size=3, padding=1)  # entrée 3 channels (RGB)
        self.conv2 = nn.Conv2d(16, 32, kernel_size=3, padding=1)
        self.conv3 = nn.Conv2d(32, 64, kernel_size=3, padding=1)
        
        self.pool = nn.MaxPool2d(2, 2)  # réduit dimension par 2 à chaque fois
        
        # Calculer la taille en sortie avant les couches fully connected
        # CIFAR10 32x32 → après 3 poolings (x2) → 4x4
        self.fc1 = nn.Linear(64 * 4 * 4, 128) # après les convolutions on a un tenseur de taille 64*4*4, fc1 les réduit en 128 neurones
        self.fc2 = nn.Linear(128, 10)  # 10 classes CIFAR10

    def forward(self, x):
        x = F.relu(self.conv1(x))
        x = self.pool(x)
        
        x = F.relu(self.conv2(x))
        x = self.pool(x)
        
        x = F.relu(self.conv3(x))
        x = self.pool(x)
        
        x = x.view(x.size(0), -1)  # aplatit en [batch_size, 64*4*4 = 1024]
        x = F.relu(self.fc1(x)) #ou fonction sigmoïde à la fin?
        x = self.fc2(x)
        return x


à entraîner (boucle d'entraînement) training loop + sauvegarde des poids à chaque epoch ; tester pour chaque epoch et voir où l'accuracy est la mieux

Masque pour le dropout

In [155]:
class CNN_MCdropout(nn.Module):
    def __init__(self, model, p1=0.1, p2=0.1, p3=0.1, p4=0.1):
        super().__init__()
        self.model = model
        self.p1 = p1  # dropout sur conv1 output
        self.p2 = p2  # dropout sur conv2 output
        self.p3 = p3  # dropout sur conv3 output
        self.p4 = p4  # dropout sur fc1 output
        # pas de dropout sur la dernière couche
    
    def dropout_mask(self, x, p):
        if not self.training or p == 0.0:
            return torch.ones_like(x)
        mask = (torch.rand_like(x) > p).float() / (1 - p) # trouver l'argument pour mettre une seed pour que ce soit reproductible
        return mask

    def forward(self, x):
        x = F.relu(self.model.conv1(x))
        x = self.dropout_mask(x, self.p1) * x
        x = self.model.pool(x)

        x = F.relu(self.model.conv2(x))
        x = self.dropout_mask(x, self.p2) * x
        x = self.model.pool(x)

        x = F.relu(self.model.conv3(x))
        x = self.dropout_mask(x, self.p3) * x
        x = self.model.pool(x)

        x = x.view(x.size(0), -1)
        x = F.relu(self.model.fc1(x))
        x = self.dropout_mask(x, self.p4) * x
        x = self.model.fc2(x)
        return x  #pas encore normalisé


Fonction d'entraînement et test

In [156]:
val_ratio = 0.1  # 10% pour validation
train_size = int((1 - val_ratio) * len(trainset))
val_size   = len(trainset) - train_size

train_subset, val_subset = random_split(trainset, [train_size, val_size]) # divise le dataset de manière aléatoire en 90/10

trainloader = torch.utils.data.DataLoader(train_subset, batch_size=batch_size, shuffle=True, num_workers=2)
valloader   = torch.utils.data.DataLoader(val_subset, batch_size=batch_size, shuffle=False, num_workers=2)

Fonction d'évaluation (avant entraînement)

In [157]:
def evaluate(model, dataloader, device):
    model.eval()
    criterion = nn.CrossEntropyLoss()
    total_loss, correct, total = 0, 0, 0
    with torch.no_grad():
        for inputs, targets in dataloader:
            inputs, targets = inputs.to(device), targets.to(device)
            outputs = model(inputs)
            total_loss += criterion(outputs, targets).item()
            _, predicted = outputs.max(1)
            total += targets.size(0)
            correct += predicted.eq(targets).sum().item()
    return total_loss/len(dataloader), correct/total

Fonction d'entraînement

In [158]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
save_path = "best.pt"

# Vérifie si les poids existent déjà
if os.path.exists(save_path):
    print("Chargement du modèle sauvegardé")
    base_model = SimpleCNN()
    base_model.load_state_dict(torch.load(save_path, map_location=device))  # même architecture que celle qui a sauvegardé
    model = CNN_MCdropout(base_model).to(device)
else:
    print("Pas de modèle sauvegardé, on entraîne le modèle")
    base_model = SimpleCNN()
    base_model = train_model(model, trainloader, valloader, device, epochs=20, save_path=save_path)
    base_model.load_state_dict(torch.load(save_path, map_location=device))  # recharge les meilleurs poids
    model = CNN_MCdropout(base_model).to(device)

# Évaluation finale
test_loss, test_acc = evaluate(model, testloader, device)
print(f"Final Test Loss: {test_loss:.4f} - Test Acc: {test_acc:.4f}")

Chargement du modèle sauvegardé
Final Test Loss: 0.8186 - Test Acc: 0.7267


In [159]:
def train_model(model, trainloader, valloader, device, epochs=20, save_path="best.pt"):
    model = model.to(device)
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=0.001)
    
    best_val_acc = 0
    
    for epoch in range(epochs):#petit nombre d'epochs pour tester (environ 1 minutes pas epoch)
        model.train()
        running_loss = 0.0
        for inputs, targets in trainloader:
            inputs, targets = inputs.to(device), targets.to(device)
            optimizer.zero_grad()
            outputs = model(inputs)
            loss = criterion(outputs, targets)
            loss.backward()
            optimizer.step()
            running_loss += loss.item()
        
        # Évaluer sur validation
        val_loss, val_acc = evaluate(model, valloader, device)
        print(f"Epoch {epoch+1}/{epochs} - Train Loss: {running_loss/len(trainloader):.4f} - Val Loss: {val_loss:.4f} - Val Acc: {val_acc:.4f}")
        
        # Sauvegarde si amélioration
        if val_acc > best_val_acc:
            best_val_acc = val_acc
            torch.save(model.state_dict(), save_path)
    
    print("Finished Training")
    return model


une fois que j'entraîne le modèle je sauvegarde les poids au format pickle, puis model_orig.loadstatedict (?) puis mettre le chemin de mes poids

In [160]:
# device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
# model = SimpleCNN()
# model = train_model(model, trainloader, valloader, device, epochs=20, save_path="best.pt") # Entraînement sur trainloader et validation sur valloader, sauvegarde du meilleur modèle

# model.load_state_dict(torch.load("best.pt", map_location=device)) # Rechargement des meilleurs poids sauvegardés

# best_model = CNN_MCdropout(model).to(device)  # best_model est juste model avec MC Dropout, model avec les meilleurs poids

# # Evaluation finale sur le test set
# test_loss, test_acc = evaluate(best_model, testloader, device)
# print(f"Final Test Loss: {test_loss:.4f} - Test Acc: {test_acc:.4f}")

MC Dropout prédiction

In [161]:
# model = best_model 

In [162]:
def mc_predict_mean_probs(model, X, T=100):
    model.train()  # activer dropout pendant l'inférence MC
    probs_list = []
    with torch.no_grad():
        for _ in range(T):
            logits_t = model(X)
            probs_t = F.softmax(logits_t, dim=1)
            probs_list.append(probs_t.unsqueeze(0))
    probs_mc = torch.cat(probs_list, dim=0)       
    model.eval()  # remettre le modèle en mode eval à la fin
    return probs_mc.mean(0)                         

In [163]:
# Test normal
test_loss, test_acc = evaluate(model, testloader, device)
print("Test acc:", test_acc)

Test acc: 0.7267


je dois garder le même batch ; mettre des seeds pour que ce soit reproductible (fonction de dropout)

In [164]:
# Test MC Dropout sur un batch
X, Y = next(iter(testloader))
X, Y = X.to(device), Y.to(device)
probs = mc_predict_mean_probs(model, X, T=100)
Y_hat = probs.argmax(1)

print("Classes prédites (MC):", Y_hat.tolist())
print("Classes vraies       :", Y.tolist())

Classes prédites (MC): [3, 8, 8, 0, 6, 6, 1, 6, 3, 1, 0, 9, 5, 7, 9, 8, 5, 7, 8, 6, 7, 2, 4, 9, 4, 2, 4, 0, 9, 6, 6, 5, 4, 3, 9, 3, 4, 1, 9, 5, 4, 6, 5, 6, 0, 9, 3, 8, 7, 6, 9, 8, 6, 3, 8, 8, 5, 5, 5, 3, 7, 5, 6, 3, 6, 6, 1, 0, 3, 9, 8, 6, 8, 8, 0, 2, 0, 3, 3, 8, 8, 1, 1, 7, 2, 9, 2, 8, 8, 9, 0, 3, 8, 6, 4, 6, 6, 0, 0, 3, 5, 5, 6, 3, 1, 1, 2, 6, 8, 5, 5, 0, 2, 2, 9, 3, 0, 4, 3, 5, 8, 3, 1, 2, 8, 0, 8, 3]
Classes vraies       : [3, 8, 8, 0, 6, 6, 1, 6, 3, 1, 0, 9, 5, 7, 9, 8, 5, 7, 8, 6, 7, 0, 4, 9, 5, 2, 4, 0, 9, 6, 6, 5, 4, 5, 9, 2, 4, 1, 9, 5, 4, 6, 5, 6, 0, 9, 3, 9, 7, 6, 9, 8, 0, 3, 8, 8, 7, 7, 4, 6, 7, 3, 6, 3, 6, 2, 1, 2, 3, 7, 2, 6, 8, 8, 0, 2, 9, 3, 3, 8, 8, 1, 1, 7, 2, 5, 2, 7, 8, 9, 0, 3, 8, 6, 4, 6, 6, 0, 0, 7, 4, 5, 6, 3, 1, 1, 3, 6, 8, 7, 4, 0, 6, 2, 1, 3, 0, 4, 2, 7, 8, 3, 1, 2, 8, 0, 8, 3]


rajouter en argument de generate_mc_outputs measure = variance (en gros on laisse à l'utilisateur le choix de la mesure d'incertitude)

In [165]:
def generate_mc_outputs(model, X, T=1000, metrics="mc_estimate", labels=None):
    model.train()  # dropout actif en inférence
    outputs = []
    with torch.no_grad():
        for _ in range(T):
            out = model(X)                      
            outputs.append(out.unsqueeze(0))    
    
    outputs = torch.cat(outputs, dim=0)         
    all_probs   = torch.softmax(outputs, dim=2)  
    mean_probs  = all_probs.mean(dim=0)          # moyenne des softmax
    var_pred  = outputs.var(dim=0)                # variance des logits
   
    results = {}
    for metric in metrics:
        if metric == "mc_estimate":
            results[metric] = mean_probs
        elif metric == "variance":
            results[metric] = var_pred
        elif metric == "predictive_entropy":
            results[metric] = -(mean_probs * (mean_probs + 1e-12).log()).sum(dim=1) # +1e-12 pour éviter log(0)
        elif metric == "relative_norm":
            if labels is None:
                raise ValueError("labels must be provided for relative_norm metric")
            labels_onehot = F.one_hot(labels, num_classes=mean_probs.size(1)).float()
            diff_norm = torch.norm(mean_probs - labels_onehot, dim=1)   # ||ŷ̄ - Y||
            denom = torch.max(torch.norm(mean_probs, dim=1), torch.norm(labels_onehot, dim=1))
            results[metric] = diff_norm / (denom + 1e-12)                  
        else:
            raise ValueError(f"Metric {metric} non reconnue")
    model.eval()  # remettre le modèle en mode eval à la fin
    return outputs, mean_probs, results


le mc estimate n'est pas une mesure d'incertitude, sert pour construire une autre variable

voir si je peux renvoyer différentes métriques à la dmd de l'utilisateur

In [168]:
# user_metric = input("Quelle métrique voulez-vous calculer ? (mc_estimate / variance / predictive_entropy / relative_norm) ")
user_metrics = input(
    "Quelles métriques voulez-vous calculer ? (mc_estimate, variance, predictive_entropy, relative_norm)\n"
    "Vous pouvez en choisir plusieurs, séparées par des virgules : "
)
user_metrics = [m.strip() for m in user_metrics.split(",")]
outputs, mean_probs, metric_values = generate_mc_outputs(model, X, T=1000, metrics=user_metrics, labels=Y)

print(f"Liste des métriques choisies par l\'utilisateur : {user_metrics}")
for metric in user_metrics:
    print(f"Métrique choisie : {metric}")
    print(f"Résultat : {metric_values[metric]}\n")

Liste des métriques choisies par l'utilisateur : ['mc_estimate', 'variance', 'predictive_entropy', 'relative_norm']
Métrique choisie : mc_estimate
Résultat : tensor([[4.6663e-04, 3.9614e-04, 1.0271e-03,  ..., 7.2859e-04, 8.5700e-03,
         3.8452e-04],
        [2.3669e-04, 3.0519e-01, 2.0701e-06,  ..., 4.4343e-08, 6.9350e-01,
         1.0700e-03],
        [1.3909e-01, 8.2842e-02, 1.1927e-03,  ..., 1.8539e-03, 7.3471e-01,
         2.5312e-02],
        ...,
        [6.0856e-01, 1.1655e-01, 1.9248e-01,  ..., 3.8263e-05, 4.6222e-02,
         2.6190e-02],
        [1.5536e-01, 3.0482e-04, 9.0639e-03,  ..., 9.3496e-05, 7.6142e-01,
         1.1015e-03],
        [1.8480e-04, 7.3716e-07, 1.3162e-03,  ..., 1.2729e-03, 3.9685e-06,
         2.3933e-04]])

Métrique choisie : variance
Résultat : tensor([[1.9079, 1.3896, 1.3107,  ..., 1.7029, 1.9423, 2.7063],
        [1.4876, 2.6907, 2.7541,  ..., 3.4867, 3.0077, 1.8900],
        [0.5950, 0.9499, 0.9361,  ..., 1.1914, 0.8998, 0.6305],
        ...,
 

pour la fonction de confiance, dois-je faire 1/métrique choisie? ou ça marche qu'avec la variance?

oui si c'est dans R+ : 1/ ? exp(-...) si c'est pour avoir entre 0 et 1 ; quitte à normaliser après